## 3. Postprocessing Proxies Workflow

0. Packages
1. Comments
2. Settings
3. Download Data & Metadata
4. Merge Data & Metadata
5. Reprojecting & Resampling
6. Clipping & Masking
7. Conversion to NetCDF

### 0. Packages

In [1]:
# Packages
import affine
import geopandas as gpd
import glob
from google.cloud import storage
import matplotlib.pyplot as plt
import netCDF4 as nc4
import numpy as np
import os
import pandas as pd
from rasterio.enums import Resampling
import rioxarray as rxr
from shapely.geometry import Polygon
from tqdm import tqdm
import xarray as xr

C:\Users\white_rn\AppData\Local\Temp\ipykernel_212\3678383558.py:3: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


### 1. Comments

Acknowledgements & code references:
- https://github.com/openearth/eo-bathymetry/
- https://github.com/openearth/eo-bathymetry-functions/
- https://github.com/gee-community/ee-packages-py

In [2]:
# TODO list
# TODO: add properties to the processed images again (CSV files from the cloud!)
# TODO: fix NC files to have correct headers & variable names
# TODO: merged product without interpolation & with interpolation (linear) over 500 m?
# TODO: standardize the NC header (interpolation flags & quality indicators)
# TODO: interpolated datasets; flag for interpolated points

### 2. Settings

In [3]:
# Settings
project_name = 'AOI_WestEurope_v2'      # Name of the project AoI, or one in the folder
mode = 'intertidal_improved_100m_upscaled_v4'  # Specify mode, either 'intertidal' or 'subtidal'
start_date = '2021-01-01'                  # Start date of the composites
stop_date = '2022-01-01'                   # End date of the composites
compo_int = 12                             # Composite interval [months]
compo_len = 12                             # Composite length [months]
scale = 100                                # Output resolution of the image [m]
crs = 'EPSG:4326'                          # Output projection of the image

upscale = 100                              # Upscaling factor for the image
res_arc_min = 1/16                         # resolution of the image in arc minutes
res_deg = res_arc_min / 60                 # resolution of the image in degrees


# Tiling (see https://www.openearth.nl/rws-bathymetry/2019.html)
zoom_levels = [9, 10, 11] # list with zoom levels
zoom_level = zoom_levels[1] # zoom level to be used

# Directories and files
dir_path_base = r'p:\11209821-cmems-global-sdb'
dir_path_output = os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', f'{mode}')                                                         # Output directory
file_path_mask = os.path.join(dir_path_base, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result.parquet')                     # Mask file
file_path_mask_ed = os.path.join(dir_path_base, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result_erosion_dilation.parquet') # Mask (erosion/dilation) file
file_path_tiles = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_polygons_world', f'df_boxes_world_Z{zoom_level}_filtered.parquet')                     # Tiles file
file_path_credentials = os.path.join(dir_path_base, '00_miscellaneous', 'KEYS', 'bathymetry-543b622ddce7.json')                                               # Cloud Storage credentials file

# Google Cloud Bucket
bucket = 'cmems-sdb'

# Load Google credentials
if not file_path_credentials == '':  
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = file_path_credentials

### 3. Download Data & Metadata

In [4]:
# Create output directories
if not os.path.exists(os.path.join(dir_path_output, '01_data')):
    os.makedirs(os.path.join(dir_path_output, '01_data'))
if not os.path.exists(os.path.join(dir_path_output, '02_metadata')):
    os.makedirs(os.path.join(dir_path_output, '02_metadata'))

# Get all files from the bucket
client = storage.Client()
all_blobs = [blob for blob in client.list_blobs(bucket)]

# Get subset of files to download
for blob in all_blobs:
    # Get mode and zoom level
    mode_blob = blob.name.split('/')[0]
    zoom_level_blob = blob.name.split('/')[1]

    # Check if mode is correct
    if mode_blob == mode:
        file_path_data = os.path.join(dir_path_output, '01_data', '_'.join(blob.name.split('/')[1:]))
    elif mode_blob == '{}_meta'.format(mode):
        file_path_data = os.path.join(dir_path_output, '02_metadata', '_'.join(blob.name.split('/')[1:]))
    else:
        continue

        # Check if zoom level is correct
    if not zoom_level_blob == f'z{zoom_level}':
        continue

    # Check if file already exists
    if os.path.exists(file_path_data):
        print(f'File already exists: {os.path.basename(file_path_data)}')
        continue

    # Download the file
    print(f'Downloading: {os.path.basename(file_path_data)}')
    blob.download_to_filename(file_path_data)

Downloading: z10_x485_y389_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x485_y392_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x486_y376_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x486_y377_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x486_y380_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x486_y386_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x486_y387_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x486_y393_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x486_y394_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x487_y377_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x487_y379_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x487_y380_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x487_y381_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x487_y382_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x487_y384_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x487_y385_t2021-01-01_2022-01-01_100m.tif
Downloading: z10_x487_y392_t2021-01-01_2022-01-01_100m.t

### 4. Merge Data & Metadata

In [5]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '01_data', '*.tif'))
file_path_csvs = glob.glob(os.path.join(dir_path_output, '02_metadata', '*.csv'))

# Number of files
print('Number of tif files: {}'.format(len(file_path_tifs)))
print('Number of csv files: {}'.format(len(file_path_csvs)))

# Merge the files
for i, file_path_tif in enumerate(tqdm(file_path_tifs)):    
    # Get the corresponding CSV file
    file_name_csv = os.path.join(dir_path_output, '02_metadata', os.path.basename(file_path_tif).replace('.tif', '.csv'))

    # Check if the CSV file exists
    if not os.path.exists(file_name_csv):
        print('CSV does not exist')
        continue
    
    # Open tif and csv files
    da = rxr.open_rasterio(file_path_tif)
    csv_data = pd.read_csv(file_name_csv, index_col=0).iloc[0].to_dict()

    # Convert gtsm_station_temporal_offsets, gtsm_tidal_stage_percentages, gtsm_water_levels, quality_scores, system_time_starts to lists
    for key in ["gtsm_station_temporal_offsets", "gtsm_tidal_stage_percentages", "gtsm_water_levels", "quality_scores", "system_time_starts"]:
        csv_data[key] = eval(csv_data[key])

    # Convert gtsm_times to datetime
    import datetime
    csv_data['gtsm_times'] = csv_data['gtsm_times'].replace('[', '[\'').replace(', ', '\', \'').replace(']', '\']')
    csv_data['gtsm_times'] = eval(csv_data['gtsm_times'])
    csv_data['gtsm_times'] = [datetime.datetime.strptime(x[:-3], '%Y-%m-%dT%H:%M:%S.%f') for x in csv_data['gtsm_times']]
    
    # Convert system_time_starts to datetime
    csv_data['system_time_starts'] = [datetime.datetime.fromtimestamp(x/1000) for x in csv_data['system_time_starts']]

    # Convert gtsm_station_temporal_offsets to deltatime 
    from datetime import timedelta
    csv_data['gtsm_station_temporal_offsets'] = [timedelta(seconds=x/1000) for x in csv_data['gtsm_station_temporal_offsets']]

    # Add the csv data as attributes to tif
    for key, value in csv_data.items():
        da.attrs[key] = value
    
    # Scale second band between 1 and 5
    band = da.band.values[1]
    band_min = da.loc[dict(band=band)].min()
    band_max = da.loc[dict(band=band)].max()
    da.loc[dict(band=band)] = ((da.loc[dict(band=band)] - band_min) / (band_max - band_min) * 4).round()
    
    # Plot the tif file
    plot = False
    if plot:
        fig, axs = plt.subplots(1, 2, figsize=(20, 10))
        da.isel(band=0).plot(ax=axs[0])
        da.isel(band=1).plot(ax=axs[1])

    # Create directory
    if not os.path.exists(os.path.join(dir_path_output, '03_merged')):
        os.makedirs(os.path.join(dir_path_output, '03_merged'))

    # Save the tif file
    file_path_merged_tif = os.path.join(dir_path_output, '03_merged', os.path.basename(file_path_tif))
    da.rio.to_raster(file_path_merged_tif, driver='GTiff', compress="LZW")
    da.close()

Number of tif files: 173
Number of csv files: 206


100%|██████████| 173/173 [00:14<00:00, 11.99it/s]


### 5. Reprojecting & Resampling

In [6]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '03_merged', '*.tif'))

# Number of files
print('Number of tif files: {}'.format(len(file_path_tifs)))

# Reproject the files
for i, file_path_tif in enumerate(tqdm(file_path_tifs)):
    # Select specific file
    #if i != 100:
    #    continue

    # Open tif file
    da = rxr.open_rasterio(file_path_tif)

    # Get bounds
    gdf_bounds = gpd.GeoDataFrame(geometry=[Polygon.from_bounds(*da.rio.bounds())], crs=da.rio.crs)

    # Reproject bounds
    gdf_bounds_reprojected = gdf_bounds.to_crs(crs)

    # Reproject the tif file
    reproject_type = 'transform'
    if reproject_type == 'scale_factor':
        scale_factor = upscale / scale
        da_reprojected = da.rio.reproject(crs, shape=(int(da.rio.height / scale_factor), int(da.rio.width / scale_factor)))
    elif reproject_type == 'resolution':
        da_reprojected = da.rio.reproject(crs, resolution=res_deg, nodata=np.nan)
    elif reproject_type == 'transform':
        shift = (0, 0) # (0.5, 0.5)
        tlx = (round(gdf_bounds_reprojected.total_bounds[0]/res_deg - shift[0]) + shift[0]) * res_deg
        tly = (round(gdf_bounds_reprojected.total_bounds[3]/res_deg - shift[1]) + shift[1]) * res_deg
        transform = affine.Affine(res_deg, 0, tlx, 0, -res_deg, tly)
        da_reprojected1 = da.isel(band=0).rio.reproject(crs, transform=transform, nodata=np.nan, resampling=Resampling.cubic)
        da_reprojected2 = da.isel(band=1).rio.reproject(crs, transform=transform, nodata=np.nan, resampling=Resampling.nearest)
        da_reprojected = xr.concat([da_reprojected1, da_reprojected2], dim='band')

    
    # Plot the tif file
    plot = False
    if plot:
        fig, axs = plt.subplots(2, 2, figsize=(20, 20))
        axs = axs.flatten()
        da.isel(band=0).plot(ax=axs[0])
        da.isel(band=1).plot(ax=axs[1])
        da_reprojected.isel(band=0).plot(ax=axs[2])
        da_reprojected.isel(band=1).plot(ax=axs[3])
        fig.tight_layout()
    
    # Check directory
    if not os.path.exists(os.path.join(dir_path_output, '04_reprojected')):
        os.makedirs(os.path.join(dir_path_output, '04_reprojected'))

    # Save the tif file
    file_path_reprojected_tif = os.path.join(dir_path_output, '04_reprojected', "_".join(os.path.basename(file_path_tif).split("_")[:-1]) + ".tif")
    da_reprojected.rio.to_raster(file_path_reprojected_tif, driver="GTiff", compress="LZW")
    da.close()
    da_reprojected.close()

Number of tif files: 173


100%|██████████| 173/173 [00:17<00:00,  9.65it/s]


### 6. Clipping & Masking

In [7]:
# Read geometries
gdf_mask = gpd.read_parquet(file_path_mask)
gdf_mask_ed = gpd.read_parquet(file_path_mask_ed)
gdf_tiles = gpd.read_parquet(file_path_tiles)

In [8]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '04_reprojected', '*.tif'))

# Sort files
file_path_tifs = sorted(file_path_tifs)

# Filter tiles based on name
names = ['_'.join(os.path.basename(x).split('_')[:3]) for x in file_path_tifs]
gdf_tiles_ss = gdf_tiles[gdf_tiles['name'].isin(names)]

# Filter files based on name
file_path_tifs = [x for x in file_path_tifs if '_'.join(os.path.basename(x).split('_')[:3]) in gdf_tiles_ss['name'].values]

# Sort tiles according to names
gdf_tiles_ss = gdf_tiles_ss.sort_values(by='name')

# Add files to tiles
gdf_tiles_ss['file_path_tif'] = file_path_tifs

# Number of files and tiles
print('Number of tif files and tiles: {}'.format(len(file_path_tifs)))

Number of tif files and tiles: 173


In [17]:
for idx, row in tqdm(gdf_tiles_ss.iterrows(), total=len(gdf_tiles_ss)):
    # Open the tif file
    da = rxr.open_rasterio(row['file_path_tif'])

    # Clip the tif file to the tile
    da_clipped = da.rio.clip([row['geometry']], da.rio.crs, drop=True)

    # Mask using histogram
    min_val = np.nanmin(da_clipped.isel(band=0).values)
    mean_val = np.nanmean(da_clipped.isel(band=0).values)
    max_val = np.nanmax(da_clipped.isel(band=0).values)
    binsize = 0.01
    bins = np.arange(min_val - binsize/2, max_val + binsize/2, binsize)
    counts, _ = np.histogram(da_clipped.isel(band=0).values[~np.isnan(da_clipped.isel(band=0).values)], bins=bins)
    bins_sorted = bins[counts.argsort()]
    count_sorted = counts[counts.argsort()]
    thresholds = [0, 0]
    thresholds[0] = bins_sorted[bins_sorted < mean_val][-1] + binsize * 2 if len(bins_sorted[bins_sorted < mean_val]) > 0 else min_val
    thresholds[1] = bins_sorted[bins_sorted > mean_val][-1] - binsize * 2 if len(bins_sorted[bins_sorted > mean_val]) > 0 else max_val
    
    da_mask = np.logical_and(da_clipped.isel(band=0) > thresholds[0], da_clipped.isel(band=0) < thresholds[-1])
    da_masked = da_clipped.where(da_mask)

    # Get the tile as a geodataframe
    gdf_tile = gdf_tiles.loc[[idx]]
    
    # Get mask that intersects with the tile
    gdf_mask_tile = gdf_mask[gdf_mask.intersects(gdf_tile.unary_union)]
    gdf_mask_ed_tile = gdf_mask_ed[gdf_mask_ed.intersects(gdf_tile.unary_union)]

    # Clip mask to the tile
    gdf_mask_tile = gpd.overlay(gdf_mask_tile, gdf_tile, how='intersection')
    gdf_mask_ed_tile = gpd.overlay(gdf_mask_ed_tile, gdf_tile, how='intersection')
    
    '''
    # Clip the tif file to the mask
    da_clipped = da_clipped.rio.clip(gdf_mask_tile.geometry, gdf_mask_tile.crs, drop=True)
    
    
    # Mask the tif file based on percentages
    da_perc_mask = np.logical_and(da_clipped.isel(band=0) > 0.95 * np.nanmin(da_clipped.isel(band=0)),
                                  da_clipped.isel(band=0) < 0.95 * np.nanmax(da_clipped.isel(band=0)))
    da_masked = da_clipped.where(da_perc_mask)
    '''

    # Plot the tif file
    plot = False
    if plot:
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        ax.bar(bins[:-1], counts, width=binsize, align='edge')
        ax.axvline(thresholds[0], color='red')
        ax.axvline(thresholds[1], color='red')
        break
        fig, axs = plt.subplots(3, 2, figsize=(30, 20))
        axs = axs.flatten()
        da.isel(band=0).plot(ax=axs[0])
        da.isel(band=1).plot(ax=axs[1])
        da_clipped.isel(band=0).plot(ax=axs[2])
        da_clipped.isel(band=1).plot(ax=axs[3])
        da_masked.isel(band=0).plot(ax=axs[4])
        da_masked.isel(band=1).plot(ax=axs[5])
        for ax in axs:
            gdf_tile.plot(ax=ax, edgecolor='green', facecolor='none')
            gdf_mask_tile.plot(ax=ax, edgecolor='red', facecolor='none')
            #gdf_mask_ed_tile.plot(ax=ax, edgecolor='purple', facecolor='none')
            ax.set_xlim(da.rio.bounds()[0], da.rio.bounds()[2])
            ax.set_ylim(da.rio.bounds()[1], da.rio.bounds()[3])
        fig.tight_layout()
    
    # Check directory
    if not os.path.exists(os.path.join(dir_path_output, '05_clipped')):
        os.makedirs(os.path.join(dir_path_output, '05_clipped'))

    # Save the tif file
    file_path_clipped_tif = os.path.join(dir_path_output, '05_clipped', os.path.basename(row['file_path_tif']))
    da_masked.rio.to_raster(file_path_clipped_tif, driver="GTiff", compress="LZW")
    da.close()
    da_clipped.close()
    da_masked.close()

100%|██████████| 173/173 [00:48<00:00,  3.57it/s]


### 7. Conversion to NetCDF

In [18]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '05_clipped', '*.tif'))

# Number of files
print('Number of tif files: {}'.format(len(file_path_tifs)))

# Loop over the files
for i, file_path_tif in tqdm(enumerate(file_path_tifs), total=len(file_path_tifs)):    
    # Open tif file
    da = rxr.open_rasterio(file_path_tif)

    # Remove spatial ref
    crs_out = da.rio.crs
    da = da.drop_vars('spatial_ref')

    # Convert data array to dataset
    da = da.assign_coords(band=['elevation', 'Quality_indicator'])
    ds = da.to_dataset(dim='band')

    # Rename x and y to lon and lat
    ds = ds.rename({'x': 'lon', 'y': 'lat'}) 

    # Add variables
    ds['SDB_type'] = xr.DataArray(np.zeros_like(ds['elevation']), dims=('lat', 'lon'), coords={'lat': ds['lat'], 'lon': ds['lon']})
    ds['crs'] = int(crs_out.to_epsg())

    # Convert types
    ds['lon'] = ds['lon'].astype('float64')
    ds['lat'] = ds['lat'].astype('float64')
    ds['elevation'] = ds['elevation'].astype('float32')
    ds['Quality_indicator'] = ds['Quality_indicator'].astype('byte')
    ds['SDB_type'] = ds['SDB_type'].astype('byte')
    ds['crs'] = ds['crs'].astype('int64')

    # Replace nan values
    ds['Quality_indicator'] = ds['Quality_indicator'].where(~np.isnan(ds['elevation']), np.byte(127))
    ds['SDB_type'] = ds['SDB_type'].where(~np.isnan(ds['elevation']), np.byte(127))
    
    # Remove attributes
    ds.lon.attrs = {'axis': 'X', 'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
    ds.lat.attrs = {'axis': 'Y', 'long_name': 'latitude', 'standard_name': 'latitude', 'units': 'degrees_north'}
    ds.elevation.attrs = {'long_name': 'Satellite derived bathymetric depth value',
                          'standard_name': 'depth',
                          'units': 'm',
                          'grid_mapping': 'crs'}
    ds.Quality_indicator.attrs = {'_FillValue': np.byte(127),
                                  'long_name': 'Unified qualitative representation of quality',
                                  'standard_name': 'quality_flag',
                                  'valid_range': [np.byte(0), np.byte(4)],
                                  'flag_values': np.array([0, 1, 2, 3, 4], dtype='byte'),
                                  'flag_meaning': 'highly_reliable Reliable Moderately_reliable Unreliable Highly_unreliable',
                                  'grid_mapping': 'crs'}
    ds.SDB_type.attrs = {'_FillValue': np.byte(127),
                         'long_name': 'SDB methodology used for elevation value',
                         'standard_name': 'SDB_methodology',
                         'valid_range': [np.byte(0), np.byte(4)],
                         'flag_values': np.array([0, 1, 2, 3, 4], dtype='byte'),
                         'flag_meaning': 'Intertidal_Bathymetry Multi_spectral_SDB Wave_kinematics Interpolated ground_truth_control_point',
                         'grid_mapping': 'crs'}
    ds.crs.attrs = {'grid_mapping_name': 'latitude_longitude', 'long_name': 'grid mapping', 'epsg_code': str(crs)}
    ds.attrs = {'AREA_OR_POINT': 'Area',
                'STATISTICS_APPROXIMATE': 'YES',
                'STATISTICS_MAXIMUM': ds['elevation'].max().values,
                'STATISTICS_MEAN': ds['elevation'].mean().values,
                'STATISTICS_MINIMUM': ds['elevation'].min().values,
                'STATISTICS_STDDEV': ds['elevation'].std().values,
                'STATISTICS_VALID_PERCENT': (ds['elevation'].count() / ds['elevation'].size * 100).values,
                'scale_factor': 1.0,
                'add_offset': 0.0,
                '_FillValue': np.nan,
                'dtm_convention_version': '1.0',
                'Conventions': 'SeaDataNet_1.0 CF1.6',
                'title': 'Provision of global coastal bathymetry derived from Sentinel 2 observations',
                'institution': 'On behalf of the Copernicus Marine Project, https://marine.copernicus.eu/',
                'source': 'Sentinel 2 observations',
                'comment': 'This data should not be used for navigation or any purpose relating to safety at sea.',
                'history': 'Created with Python script in January 2025'}

    # Check directory
    if not os.path.exists(os.path.join(dir_path_output, '06_netcdf')):
        os.makedirs(os.path.join(dir_path_output, '06_netcdf'))

    # Write to netcdf
    file_path_netcdf = os.path.join(dir_path_output, '06_netcdf', os.path.basename(file_path_tif).replace('.tif', '.nc'))
    ds.to_netcdf(file_path_netcdf, mode="w", format='NetCDF4')
    
    # Read netcdf
    cdf = False
    if cdf:
        with nc4.Dataset(file_path_netcdf) as nc:
            # Write header file (cdl)
            file_path_cdl = file_path_netcdf.replace(".nc", ".cdl")
            cdl = nc.tocdl(file_path_cdl)
            with open(file_path_cdl, 'w') as f:
                f.write(cdl)

Number of tif files: 173


  0%|          | 0/173 [00:00<?, ?it/s]

100%|██████████| 173/173 [00:18<00:00,  9.21it/s]
